# Edit Producer

Runs editing and model generation at each milestone. This notebook writes `unscored/`, `time/`, and `mem/` files only. Run `eval.ipynb` in another process/kernel to judge the unscored snapshots and rebuild `perf/` results.

## Environment

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"edit-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"


def log(message: str):
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as file:
        file.write(f"{line}\n")

## Evaluation Config

In [ ]:
import copy

import yaml

def resolve_run_indices(n_runs: int, run_start: int = 1, run_count: int | None = None) -> list[int]:
    if n_runs <= 0:
        raise ValueError("n_runs must be positive")
    if run_start <= 0:
        raise ValueError("run_start must be positive")

    if run_count is None:
        run_end = n_runs
    else:
        run_count = int(run_count)
        if run_count <= 0:
            raise ValueError("run_count must be positive when set")
        run_end = run_start + run_count - 1

    if run_end > n_runs:
        raise ValueError(
            f"run_start/run_count select runs {run_start}..{run_end}, "
            f"but n_runs={n_runs}"
        )

    return list(range(run_start, run_end + 1))


EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

editing_methods = list(eval_config["editing_methods"])
model_name = eval_config["model_name"]
result_names = list(eval_config["result_names"])
eval_milestones = list(eval_config["eval_milestones"])
n_runs = int(eval_config["n_runs"])
run_start = int(eval_config.get("run_start", 1))
raw_run_count = eval_config.get("run_count")
run_count = None if raw_run_count is None else int(raw_run_count)
run_indices = resolve_run_indices(n_runs, run_start, run_count)
judge_repeats = int(eval_config["judge"]["repeats"])
base_seed = int(eval_config["base_seed"])

generation_config = copy.deepcopy(eval_config["generation"])

set_seed(base_seed)
log(f"[config] loaded={EVAL_CONFIG_PATH} model={model_name} results={result_names} runs={run_indices}")

## Data Loading And Shared Helpers

In [ ]:
from collections.abc import Callable
from itertools import chain
from random import Random
from statistics import mean

from datasets import load_dataset

from src.editors import EditRequest
from src.evaluate import EvalRecord, EvalResult, MetricResult


es_loc_data = load_dataset("Savassan/KnowledgeSurgeon", "es_loc", split="data") if "es_loc" in result_names else None
port_data = load_dataset("Savassan/KnowledgeSurgeon", "port", split="data") if "port" in result_names else None


def build_inputs(
    dataset,
    build_data: Callable[[object], tuple[EditRequest, EvalResult]],
    seed: int,
) -> tuple[list[EditRequest], list[EvalResult]]:

    n_records = eval_milestones[-1]

    if n_records <= 0:
        raise ValueError("n_records must be > 0")
    if n_records > len(dataset):
        raise ValueError(f"Requested {n_records} records, but only {len(dataset)} records are available")

    selected = list(dataset)
    Random(seed).shuffle(selected)
    selected = selected[:n_records]

    built = [build_data(record) for record in selected]
    edit_requests = [item[0] for item in built]
    eval_results = [item[1] for item in built]
    return edit_requests, eval_results


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def _aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None

        return MetricResult(
            score=mean(metric.score for metric in metric_results),
            details=list(chain.from_iterable(metric.details for metric in metric_results)),
        )

    return EvalResult(
        success=_aggregate_metric("success"),
        locality=_aggregate_metric("locality"),
        portability=_aggregate_metric("portability"),
    )


def format_elapsed(seconds: float) -> str:
    total_seconds = int(round(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

## Benchmark Engine

In [ ]:
import asyncio
import copy
import gc
from dataclasses import asdict
from time import perf_counter
from typing import Any

from tqdm.notebook import trange

from src.editors import Editor, EditRequest, load_editor
from src.evaluate import (
    EvalRecord, EvalResult,
    generate_predictions,
    save_unscored_results,
    save_time_results, save_memory_results,
)


GLOBAL_MODEL = None
GLOBAL_TOKENIZER = None

### CUDA Memory Helpers

In [ ]:
def normalize_cuda_device_indices(device_indices: list[int] | None = None) -> list[int]:
    if not torch.cuda.is_available():
        return []
    if device_indices is None:
        return list(range(torch.cuda.device_count()))
    return [idx for idx in device_indices if idx is not None and 0 <= idx < torch.cuda.device_count()]


def reset_cuda_peak_memory(device_indices: list[int] | None = None):
    if not torch.cuda.is_available():
        return
    for device_idx in normalize_cuda_device_indices(device_indices):
        torch.cuda.reset_peak_memory_stats(device_idx)


def get_cuda_peak_memory(device_indices: list[int] | None = None):
    if not torch.cuda.is_available():
        return None

    devices = []
    total_allocated = 0
    total_reserved = 0
    for device_idx in normalize_cuda_device_indices(device_indices):
        allocated = torch.cuda.max_memory_allocated(device_idx)
        reserved = torch.cuda.max_memory_reserved(device_idx)
        total_allocated += allocated
        total_reserved += reserved
        devices.append({
            "index": device_idx,
            "peak_allocated_gb": allocated / (1024 ** 3),
            "peak_reserved_gb": reserved / (1024 ** 3),
        })

    return {
        "devices": devices,
        "total_peak_allocated_gb": total_allocated / (1024 ** 3),
        "total_peak_reserved_gb": total_reserved / (1024 ** 3),
    }


def synchronize_cuda_devices(device_indices: list[int] | None = None):
    if not torch.cuda.is_available():
        return
    for device_idx in normalize_cuda_device_indices(device_indices):
        torch.cuda.synchronize(device_idx)


def cleanup_cuda_cache(device_indices: list[int] | None = None):
    gc.collect()
    if not torch.cuda.is_available():
        return
    current_device = torch.cuda.current_device()
    try:
        for device_idx in normalize_cuda_device_indices(device_indices):
            with torch.cuda.device(device_idx):
                torch.cuda.empty_cache()
    finally:
        torch.cuda.set_device(current_device)


def merge_peak_memory(current, candidate):
    if candidate is None:
        return current
    if current is None:
        return candidate

    return candidate if candidate["total_peak_reserved_gb"] > current["total_peak_reserved_gb"] else current

### Generation Snapshot Helpers

In [ ]:
def get_editor_config_path(editing_method: str) -> Path:
    return PROJECT_ROOT / "src" / "config" / editing_method / f"{model_name}.yaml"


def get_generation_runtime_config(desc: str, editing_method: str | None = None) -> dict[str, Any]:
    runtime_config = copy.deepcopy(generation_config)
    method_overrides = runtime_config.pop("method_overrides", {}) or {}
    runtime_config.pop("async_generation", None)
    if editing_method in method_overrides:
        override = copy.deepcopy(method_overrides[editing_method] or {})
        override_model_config = override.pop("model_config", None)
        if override_model_config:
            runtime_config.setdefault("model_config", {}).update(override_model_config)
        runtime_config.update(override)
    runtime_config["desc"] = desc
    return runtime_config


def snapshot_eval_inputs(
    eval_results: list[EvalResult],
    step: int,
) -> tuple[list[EvalResult], list[list[EvalRecord]]]:
    snapshot_results = copy.deepcopy(eval_results[:step])
    snapshot_records: list[list[EvalRecord]] = []
    for result in snapshot_results:
        details: list[EvalRecord] = []
        for metric_result in (result.success, result.locality, result.portability):
            if metric_result is not None:
                details.extend(metric_result.details)
        snapshot_records.append(details)
    return snapshot_results, snapshot_records


def generate_and_save_unscored(
    model,
    tokenizer,
    eval_results: list[EvalResult],
    edit_requests: list[EditRequest],
    editing_method: str,
    result_name: str,
    step: int,
    max_step: int,
    run_index: int,
    generation_device_idx: int | None = None,
) -> Path:
    if generation_device_idx is not None and torch.cuda.is_available():
        torch.cuda.set_device(generation_device_idx)

    step_eval_results, step_records = snapshot_eval_inputs(eval_results, step)
    generation_step_config = get_generation_runtime_config(
        f"Generation [{editing_method} @ {step}]",
        editing_method=editing_method,
    )
    generation_records = sum(len(record_group) for record_group in step_records)
    batch_size = generation_step_config.get("batch_size", 32)

    log(
        "[generation-running] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method} "
        f"result={result_name} "
        f"step={step}/{max_step} "
        f"device={generation_device_idx if generation_device_idx is not None else 'edit'} "
        f"records={generation_records} "
        f"batch_size={batch_size}"
    )
    generate_predictions(
        model,
        tokenizer,
        step_eval_results,
        step_records,
        **generation_step_config,
    )
    step_result = aggregate_eval_results(step_eval_results)
    unscored_payload = {
        "step": step,
        "edits": [
            {
                "prompt": req.prompt,
                "old_ground_truth": req.old_ground_truth,
                "new_ground_truth": req.new_ground_truth,
            }
            for req in edit_requests[:step]
        ],
        "result": asdict(step_result),
    }
    unscored_path = save_unscored_results(
        result=unscored_payload,
        edited_model=model_name,
        editing_method=editing_method,
        milestones=eval_milestones,
        result_name=result_name,
        run_index=run_index,
        step=step,
        n_runs=n_runs,
        judge_repeats=judge_repeats,
    )
    log(
        "[milestone-generated] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method} "
        f"result={result_name} "
        f"step={step}/{max_step} "
        f"device={generation_device_idx if generation_device_idx is not None else 'edit'} "
        f"unscored={unscored_path}"
    )
    return unscored_path

### Async Generation Devices

In [ ]:
def parse_cuda_device_index(value) -> int:
    if isinstance(value, int):
        return value
    text = str(value).strip().lower()
    if text.isdigit():
        return int(text)
    if text.startswith("cuda:"):
        return int(text.split(":", 1)[1])
    raise ValueError(f"Unsupported CUDA device spec: {value!r}")


def model_device_index(model) -> int | None:
    for param in model.parameters():
        if param.device.type == "cuda":
            return param.device.index
    return None


def async_generation_device_indices(editor: Editor, editing_method: str) -> list[int]:
    async_config = generation_config.get("async_generation", {}) or {}
    enabled = async_config.get("enabled", "auto")
    if isinstance(enabled, str):
        enabled_key = enabled.lower()
    else:
        enabled_key = "true" if enabled else "false"

    if enabled_key in {"false", "0", "no", "off"}:
        return []
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        if enabled_key == "true":
            raise RuntimeError("Async generation requires at least 2 CUDA devices")
        return []

    unsafe_methods = set(async_config.get("unsafe_methods", []))
    if editing_method in unsafe_methods:
        log(f"[async-generation-disabled] method={editing_method} reason=stateful_adapter")
        return []

    edit_device_idx = model_device_index(editor.model)
    devices = async_config.get("devices", "auto")
    if devices == "auto":
        candidate_indices = list(range(torch.cuda.device_count()))
    else:
        candidate_indices = [parse_cuda_device_index(device) for device in devices]

    generation_indices = [idx for idx in candidate_indices if idx != edit_device_idx]
    generation_indices = [idx for idx in generation_indices if 0 <= idx < torch.cuda.device_count()]

    if not generation_indices and enabled_key == "true":
        raise RuntimeError("No valid generation CUDA devices are available")

    return generation_indices


def sync_editor_state(source_editor: Editor, target_editor: Editor, target_device_idx: int) -> None:
    source_editor.sync_generation_state_to(target_editor)
    target_editor.model.eval()
    if torch.cuda.is_available():
        source_idx = model_device_index(source_editor.model)
        if source_idx is not None:
            torch.cuda.synchronize(source_idx)
        torch.cuda.synchronize(target_device_idx)

### Async Generation Manager

In [ ]:
class AsyncGenerationManager:
    def __init__(
        self,
        editor: Editor,
        editing_method: str,
        result_name: str,
        run_index: int,
        max_step: int,
        config_path: Path,
        generation_device_indices: list[int],
    ):
        self.editor = editor
        self.editing_method = editing_method
        self.result_name = result_name
        self.run_index = run_index
        self.max_step = max_step
        self.slots: list[dict[str, Any]] = []

        for device_idx in generation_device_indices:
            log(
                "[async-generation-load] "
                f"run={run_index}/{n_runs} "
                f"method={editing_method} "
                f"device=cuda:{device_idx}"
            )
            gen_editor = load_editor(str(config_path), gpu_id=device_idx)
            gen_editor.model.eval()
            self.slots.append({
                "device_idx": device_idx,
                "editor": gen_editor,
                "task": None,
            })

    @property
    def enabled(self) -> bool:
        return bool(self.slots)

    async def submit(
        self,
        step: int,
        edit_requests: list[EditRequest],
        eval_results: list[EvalResult],
    ) -> None:
        slot = await self._acquire_slot(step)
        device_idx = slot["device_idx"]
        log(
            "[milestone-sync-start] "
            f"run={self.run_index}/{n_runs} "
            f"method={self.editing_method} "
            f"result={self.result_name} "
            f"step={step}/{self.max_step} "
            f"device=cuda:{device_idx}"
        )
        sync_editor_state(self.editor, slot["editor"], device_idx)
        log(
            "[milestone-sync-done] "
            f"run={self.run_index}/{n_runs} "
            f"method={self.editing_method} "
            f"result={self.result_name} "
            f"step={step}/{self.max_step} "
            f"device=cuda:{device_idx}"
        )

        loop = asyncio.get_running_loop()
        task = loop.run_in_executor(
            None,
            self._build_generation_job(slot, step, edit_requests, eval_results),
        )

        def log_generation_task_error(done_task: asyncio.Task) -> None:
            if done_task.cancelled():
                log(
                    "[generation-cancelled] "
                    f"run={self.run_index}/{n_runs} "
                    f"method={self.editing_method} "
                    f"result={self.result_name} "
                    f"step={step}/{self.max_step} "
                    f"device=cuda:{device_idx}"
                )
                return
            exc = done_task.exception()
            if exc is not None:
                log(
                    "[generation-error] "
                    f"run={self.run_index}/{n_runs} "
                    f"method={self.editing_method} "
                    f"result={self.result_name} "
                    f"step={step}/{self.max_step} "
                    f"device=cuda:{device_idx} "
                    f"error={type(exc).__name__}: {exc}"
                )

        task.add_done_callback(log_generation_task_error)
        slot["task"] = task
        log(
            "[generation-start] "
            f"run={self.run_index}/{n_runs} "
            f"method={self.editing_method} "
            f"result={self.result_name} "
            f"step={step}/{self.max_step} "
            f"device=cuda:{device_idx}"
        )

    async def close(self) -> None:
        while self._active_tasks():
            await self._wait_for_one()

        for slot in self.slots:
            try:
                slot["editor"].reset()
            except Exception as exc:
                log(
                    "[async-generation-reset-error] "
                    f"method={self.editing_method} "
                    f"device=cuda:{slot['device_idx']} "
                    f"error={type(exc).__name__}: {exc}"
                )
            del slot["editor"]
        self.slots.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _active_tasks(self) -> list[asyncio.Future]:
        return [slot["task"] for slot in self.slots if slot.get("task") is not None]

    async def _acquire_slot(self, step: int) -> dict[str, Any]:
        while True:
            for slot in self.slots:
                task = slot.get("task")
                if task is None:
                    return slot
                if task.done():
                    await self._finish_slot(slot)
                    return slot

            log(
                "[milestone-wait] "
                f"run={self.run_index}/{n_runs} "
                f"method={self.editing_method} "
                f"result={self.result_name} "
                f"step={step}/{self.max_step} "
                "reason=generation_devices_busy"
            )
            await self._wait_for_one()

    async def _wait_for_one(self) -> None:
        active_tasks = self._active_tasks()
        if not active_tasks:
            return
        done_tasks, _ = await asyncio.wait(active_tasks, return_when=asyncio.FIRST_COMPLETED)
        for done_task in done_tasks:
            for slot in self.slots:
                if slot.get("task") is done_task:
                    await self._finish_slot(slot)
                    break

    async def _finish_slot(self, slot: dict[str, Any]) -> None:
        task = slot.get("task")
        if task is None:
            return
        slot["task"] = None
        try:
            path = await task
        except Exception as exc:
            log(
                "[generation-failed] "
                f"run={self.run_index}/{n_runs} "
                f"method={self.editing_method} "
                f"result={self.result_name} "
                f"device=cuda:{slot['device_idx']} "
                f"error={type(exc).__name__}: {exc}"
            )
            raise
        log(
            "[generation-done] "
            f"run={self.run_index}/{n_runs} "
            f"method={self.editing_method} "
            f"result={self.result_name} "
            f"device=cuda:{slot['device_idx']} "
            f"unscored={path}"
        )

    def _build_generation_job(
        self,
        slot: dict[str, Any],
        step: int,
        edit_requests: list[EditRequest],
        eval_results: list[EvalResult],
    ):
        device_idx = slot["device_idx"]

        def _job() -> Path:
            return generate_and_save_unscored(
                model=slot["editor"].model,
                tokenizer=slot["editor"].tokenizer,
                eval_results=eval_results,
                edit_requests=edit_requests,
                editing_method=self.editing_method,
                result_name=self.result_name,
                step=step,
                max_step=self.max_step,
                run_index=self.run_index,
                generation_device_idx=device_idx,
            )

        return _job

### Method Benchmark Loop

In [ ]:
async def run_method_benchmark(
    editor: Editor,
    edit_requests: list[EditRequest],
    eval_results: list[EvalResult],
    editing_method: str,
    result_name: str,
    max_step: int,
    run_index: int,
    config_path: Path,
) -> tuple[list[float], dict | None]:

    log(
        f"[method-start] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method} "
        f"result={result_name}"
    )

    run_start = perf_counter()
    total_edit_time = 0.0
    edit_times: list[float] = []
    milestones = set(eval_milestones)
    edit_peak_memory = None
    generation_manager: AsyncGenerationManager | None = None
    edit_device_idx = model_device_index(editor.model)
    edit_memory_devices = [edit_device_idx] if edit_device_idx is not None else None

    generation_devices = async_generation_device_indices(editor, editing_method)
    if generation_devices:
        generation_manager = AsyncGenerationManager(
            editor=editor,
            editing_method=editing_method,
            result_name=result_name,
            run_index=run_index,
            max_step=max_step,
            config_path=config_path,
            generation_device_indices=generation_devices,
        )
        log(
            "[async-generation-enabled] "
            f"run={run_index}/{n_runs} "
            f"method={editing_method} "
            f"devices={generation_devices}"
        )
    else:
        log(
            "[async-generation-disabled] "
            f"run={run_index}/{n_runs} "
            f"method={editing_method} "
            "reason=no_usable_generation_device"
        )

    try:
        for step in trange(1, max_step + 1, desc=f"Benchmark [{editing_method}]", leave=False):
            reset_cuda_peak_memory(edit_memory_devices)
            synchronize_cuda_devices(edit_memory_devices)
            edit_start = perf_counter()
            editor.edit(edit_requests[step - 1])
            synchronize_cuda_devices(edit_memory_devices)
            edit_time = perf_counter() - edit_start
            edit_peak_memory = merge_peak_memory(edit_peak_memory, get_cuda_peak_memory(edit_memory_devices))
            edit_times.append(edit_time)
            total_edit_time += edit_time

            if step not in milestones:
                continue

            if generation_manager is not None and generation_manager.enabled:
                await generation_manager.submit(step, edit_requests, eval_results)
            else:
                generate_and_save_unscored(
                    model=editor.model,
                    tokenizer=editor.tokenizer,
                    eval_results=eval_results,
                    edit_requests=edit_requests,
                    editing_method=editing_method,
                    result_name=result_name,
                    step=step,
                    max_step=max_step,
                    run_index=run_index,
                )
                cleanup_cuda_cache(edit_memory_devices)
    finally:
        if generation_manager is not None:
            await generation_manager.close()

    avg_edit_time = total_edit_time / max_step
    elapsed = perf_counter() - run_start

    log(
        "[method-end] "
        f"run={run_index}/{n_runs} "
        f"method={editing_method} "
        f"result={result_name} "
        f"avg_edit_time={avg_edit_time:.2f}s "
        f"total_edit_time={format_elapsed(total_edit_time)} "
        f"elapsed={format_elapsed(elapsed)}"
    )

    return edit_times, edit_peak_memory

### Result Benchmark Loop

In [ ]:
async def run_benchmark(
    result_name: str,
    dataset,
    build_data: Callable[[object], tuple[EditRequest, EvalResult]],
    save_time: bool = True,
) -> None:

    if not eval_milestones:
        raise ValueError("eval_milestones must not be empty")
    if eval_milestones[0] <= 0:
        raise ValueError("Milestones must be positive integers")

    global GLOBAL_MODEL, GLOBAL_TOKENIZER
    max_step = eval_milestones[-1]

    log(
        "[run-start] "
        f"methods={editing_methods} "
        f"model={model_name} "
        f"result={result_name} "
        f"milestones={eval_milestones} "
        f"r{n_runs}-j{judge_repeats} "
        f"generation_batch_size={generation_config.get('batch_size', 32)} "
    )

    dataset_start = perf_counter()
    for run_index in run_indices:
        run_seed = base_seed + run_index - 1
        edit_requests, eval_results = build_inputs(dataset, build_data, run_seed)

        log(
            "[run-sample] "
            f"run={run_index}/{n_runs} "
            f"seed={run_seed}"
        )

        for editing_method in editing_methods:
            config_path = get_editor_config_path(editing_method)
            editor = load_editor(
                str(config_path),
                model=GLOBAL_MODEL,
                tokenizer=GLOBAL_TOKENIZER,
            )
            try:
                if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
                    GLOBAL_MODEL = editor.model
                    GLOBAL_TOKENIZER = editor.tokenizer

                edit_times, edit_peak_memory = await run_method_benchmark(
                    editor=editor,
                    edit_requests=edit_requests,
                    eval_results=eval_results,
                    editing_method=editing_method,
                    result_name=result_name,
                    max_step=max_step,
                    run_index=run_index,
                    config_path=config_path,
                )

                if save_time:
                    save_time_results(
                        edit_times=edit_times,
                        edited_model=model_name,
                        editing_method=editing_method,
                        milestones=eval_milestones,
                        result_name=result_name,
                        run_index=run_index,
                        n_runs=n_runs,
                        judge_repeats=judge_repeats,
                    )
                    save_memory_results(
                        memory={"edit": edit_peak_memory},
                        edited_model=model_name,
                        editing_method=editing_method,
                        milestones=eval_milestones,
                        result_name=result_name,
                        run_index=run_index,
                        n_runs=n_runs,
                        judge_repeats=judge_repeats,
                    )
            finally:
                editor.reset()
                del editor

    elapsed = perf_counter() - dataset_start

    log(
        "[run-end] "
        f"methods={editing_methods} "
        f"model={model_name} "
        f"result={result_name} "
        f"milestones={eval_milestones} "
        f"r{n_runs}-j{judge_repeats} "
        f"generation_batch_size={generation_config.get('batch_size', 32)} "
        f"elapsed={format_elapsed(elapsed)}"
    )

## ES And Locality Run

In [ ]:
def build_es_loc_data(record) -> tuple[EditRequest, EvalResult]:
    edit = record["edit"]
    answer = edit["answer"]
    edit_request = EditRequest(edit["subject"], edit["prompt"], answer["true"], answer["new"])

    es_records = [EvalRecord(edit["prompt"], [answer["new"]])]\
               + [EvalRecord(prompt, [answer["new"]]) for prompt in record["generality"]]
    loc_records = [EvalRecord(item["prompt"], [item["answer"]])
                   for item in record["locality"]]

    eval_result = EvalResult(
        success=MetricResult(details=es_records),
        locality=MetricResult(details=loc_records),
    )

    return edit_request, eval_result


if "es_loc" in result_names:
    await run_benchmark("es_loc", es_loc_data, build_es_loc_data, save_time=True)
else:
    log("[skip] result=es_loc is not enabled in result_names")

## Portability Run

In [ ]:
def build_port_data(record) -> tuple[EditRequest, EvalResult]:
    edit = record["edit"]
    answer = edit["answer"]
    edit_request = EditRequest(
        edit["subject"]["str"],
        edit["prompt"],
        answer["true"]["str"],
        answer["new"]["str"],
    )

    port_records = [
        EvalRecord(prompt, record["portability"]["answers"]["new"])
        for prompt in record["portability"]["prompts"]
    ]

    eval_result = EvalResult(
        portability=MetricResult(details=port_records)
    )

    return edit_request, eval_result


if "port" in result_names:
    await run_benchmark("port", port_data, build_port_data, save_time=True)
else:
    log("[skip] result=port is not enabled in result_names")